# Лабораторная работа: настройка и применение MLflow

**Тема:** Трекинг экспериментов, сравнение запусков обучения и разворачивание модели с помощью MLflow

**Датасет:** California Housing Prices — предсказание медианной стоимости жилья в Калифорнии по характеристикам района (регрессия).
Источник: `https://raw.githubusercontent.com/sonarsushant/California-House-Price-Prediction/master/housing.csv`


In [1]:
import pandas as pd
import numpy as np
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, PowerTransformer
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
from mlflow.models import infer_signature

## 1. Настройка MLflow




In [2]:
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("california_housing_price")

2026/06/19 08:25:15 INFO mlflow.tracking.fluent: Experiment with name 'california_housing_price' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///home/egor/mlops_labs/lab3_mlflow/mlruns/975309354776102935', creation_time=1781839515892, effective_trace_archival_retention=None, experiment_id='975309354776102935', last_update_time=1781839515892, lifecycle_stage='active', name='california_housing_price', tags={}, trace_location=None, workspace='default'>

## 2. Загрузка и первичный анализ данных

Датасет содержит 20640 наблюдений и 10 столбцов: 9 признаков (8 числовых + 1 категориальный
`ocean_proximity` — близость к океану) и целевая переменная `median_house_value` — медианная стоимость
домов в районе (в долларах).

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/sonarsushant/California-House-Price-Prediction/master/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [5]:
print("Пропуски по столбцам:")
print(df.isna().sum())
print()
print("Распределение категориального признака ocean_proximity:")
print(df['ocean_proximity'].value_counts())

Пропуски по столбцам:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

Распределение категориального признака ocean_proximity:
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64


In [6]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


## 3. Предобработка данных

Анализ гистограмм + здравый смысл:

- **пропуски** в `total_bedrooms` (207 строк) заполняем медианой — признак числовой, а доля пропусков мала (~1%);
- **мусорные строки** с нулевым числом комнат/населения/домохозяйств убираем (физически бессмысленные значения);
- целевая переменная `median_house_value` в этом датасете **искусственно обрезана** сверху на уровне 500001$
  (известная особенность сбора данных переписи США — цензурирование). Такие строки убираем, чтобы не искажать обучение;
- категориальный признак `ocean_proximity` кодируем порядковым кодированием (`OrdinalEncoder`), аналогично примеру;
- числовые признаки масштабируем `StandardScaler`, целевую переменную — `PowerTransformer`
  (стабилизирует дисперсию, у `median_house_value` распределение скошено).

In [7]:
CAT_COLUMNS = ['ocean_proximity']
TARGET = 'median_house_value'


def preprocessing_data_frame(frame):
    df = frame.copy()

    # заполняем пропуски в числовом признаке медианой
    df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

    # здравый смысл: уберём строки с нулевым количеством комнат/населения/домохозяйств
    df = df[(df['total_rooms'] > 0) & (df['population'] > 0) & (df['households'] > 0)]

    # известная особенность датасета: целевая переменная "обрезана" сверху на уровне 500001 -
    # это артефакт сбора данных (цензурирование), а не реальная цена, поэтому такие строки убираем
    df = df[df[TARGET] < df[TARGET].max()]

    df = df.reset_index(drop=True)

    # порядковое кодирование категориального признака
    ordinal = OrdinalEncoder()
    df[CAT_COLUMNS] = ordinal.fit_transform(df[CAT_COLUMNS])
    return df


def scale_frame(frame):
    df = frame.copy()
    X, y = df.drop(columns=[TARGET]), df[TARGET]
    scaler = StandardScaler()
    power_trans = PowerTransformer()
    X_scale = scaler.fit_transform(X.values)
    y_scale = power_trans.fit_transform(y.values.reshape(-1, 1))
    return X_scale, y_scale, power_trans, scaler

In [8]:
df_proc = preprocessing_data_frame(df)
X, Y, power_trans, scaler = scale_frame(df_proc)

# разбиваем на тренировочную и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.3, random_state=42)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (13772, 9), Val: (5903, 9)


## 4. Обучение моделей с трекингом в MLflow

Для содержательного сравнения обучаем **три семейства моделей**, типичных для табличной регрессии:

- **линейная модель** — `SGDRegressor` (быстрая линейная регрессия с регуляризацией);
- **дерево решений** — `DecisionTreeRegressor`;
- **случайный лес** — `RandomForestRegressor` (ансамбль деревьев).

Для каждого семейства перебираем несколько комбинаций гиперпараметров — каждая комбинация логируется
как отдельный запуск (run) в MLflow с параметрами, метриками (`rmse`, `mae`, `r2`, время обучения) и
самой моделью (артефакт). Так в интерфейсе MLflow можно сравнивать и сортировать все запуски сразу.

Параметр `model` (`linear` / `tree` / `forest`) логируется отдельно — по нему можно фильтровать запуски
в UI, например: `params.model = "tree"`.

In [9]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2


def run_experiment(model_name, model, params):
    run_name = f"{model_name}_" + "_".join(str(v) for v in params.values())
    with mlflow.start_run(run_name=run_name):
        t0 = time.time()
        model.set_params(**params)
        model.fit(X_train, y_train.reshape(-1))
        train_time = time.time() - t0

        y_pred = model.predict(X_val)
        y_price_pred = power_trans.inverse_transform(y_pred.reshape(-1, 1))
        y_price_true = power_trans.inverse_transform(y_val)
        rmse, mae, r2 = eval_metrics(y_price_true, y_price_pred)

        mlflow.log_param("model", model_name)
        for k, v in params.items():
            mlflow.log_param(k, v)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("train_time_sec", train_time)

        predictions = model.predict(X_train)
        signature = infer_signature(X_train, predictions)
        mlflow.sklearn.log_model(model, "model", signature=signature)

        run_id = mlflow.active_run().info.run_id
        print(f"{model_name:8s} {params} -> rmse={rmse:9.1f} mae={mae:9.1f} r2={r2:.4f} t={train_time:5.2f}s  run_id={run_id}")
        return run_id

### 4.1. Линейная модель (SGDRegressor)

In [10]:
linear_grid = [
    {"alpha": 0.0001, "l1_ratio": 0.001},
    {"alpha": 0.0001, "l1_ratio": 0.15},
    {"alpha": 0.001,  "l1_ratio": 0.001},
    {"alpha": 0.01,   "l1_ratio": 0.15},
]
for params in linear_grid:
    run_experiment("linear", SGDRegressor(random_state=42, max_iter=2000), params)

2026/06/19 08:25:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/19 08:25:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


linear   {'alpha': 0.0001, 'l1_ratio': 0.001} -> rmse=  63860.8 mae=  44367.3 r2=0.5852 t= 0.01s  run_id=a6e36df15f7d4a8d8b000b0d953cced2


2026/06/19 08:25:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


linear   {'alpha': 0.0001, 'l1_ratio': 0.15} -> rmse=  63860.8 mae=  44367.3 r2=0.5852 t= 0.02s  run_id=f27d7d90474e4d4c8c3c48a61abf7232


2026/06/19 08:25:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


linear   {'alpha': 0.001, 'l1_ratio': 0.001} -> rmse=  63888.1 mae=  44393.4 r2=0.5848 t= 0.01s  run_id=103b9241ee5f417f98eb1232cdfa20e2
linear   {'alpha': 0.01, 'l1_ratio': 0.15} -> rmse=  64225.4 mae=  44719.7 r2=0.5804 t= 0.01s  run_id=dbfd470eba3b4e59a3acd14ce12b423b


### 4.2. Дерево решений (DecisionTreeRegressor)

In [11]:
tree_grid = [
    {"max_depth": 4,  "min_samples_leaf": 5},
    {"max_depth": 8,  "min_samples_leaf": 5},
    {"max_depth": 12, "min_samples_leaf": 5},
    {"max_depth": 20, "min_samples_leaf": 2},
]
for params in tree_grid:
    run_experiment("tree", DecisionTreeRegressor(random_state=42), params)

2026/06/19 08:25:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/19 08:25:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


tree     {'max_depth': 4, 'min_samples_leaf': 5} -> rmse=  66943.7 mae=  47914.9 r2=0.5442 t= 0.04s  run_id=b2574a6b54ad4fa7925f0d0b9fcd9d65


2026/06/19 08:25:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


tree     {'max_depth': 8, 'min_samples_leaf': 5} -> rmse=  54500.4 mae=  37546.5 r2=0.6979 t= 0.06s  run_id=04133a070e824364970f8ca5f5fda720


2026/06/19 08:25:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


tree     {'max_depth': 12, 'min_samples_leaf': 5} -> rmse=  53519.9 mae=  35990.2 r2=0.7086 t= 0.16s  run_id=38733e02defb45abacf16e135c6e03f4
tree     {'max_depth': 20, 'min_samples_leaf': 2} -> rmse=  59305.6 mae=  39597.1 r2=0.6422 t= 0.10s  run_id=0c64d78d7cf04ff08a22092f8be5c5ea


### 4.3. Случайный лес (RandomForestRegressor)

In [12]:
forest_grid = [
    {"n_estimators": 100, "max_depth": 12},
    {"n_estimators": 200, "max_depth": 20},
    {"n_estimators": 200, "max_depth": None},
]
for params in forest_grid:
    run_experiment("forest", RandomForestRegressor(random_state=42, n_jobs=-1), params)

2026/06/19 08:26:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


forest   {'n_estimators': 100, 'max_depth': 12} -> rmse=  46728.5 mae=  31086.9 r2=0.7779 t= 1.91s  run_id=fb0381ea76964e1faa4ffb4659e8e47a


2026/06/19 08:26:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


forest   {'n_estimators': 200, 'max_depth': 20} -> rmse=  45246.1 mae=  29633.9 r2=0.7918 t= 4.54s  run_id=512c65ea5e144c56b5e12d44fce782b1


2026/06/19 08:26:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


forest   {'n_estimators': 200, 'max_depth': None} -> rmse=  45211.3 mae=  29593.4 r2=0.7921 t= 4.88s  run_id=71d05e9695704f4db73485e03111b480


## 5. Сравнение запусков

Получаем все запуски текущего эксперимента через `mlflow.search_runs()` и сортируем по метрике `r2`
(чем выше — тем лучше модель объясняет дисперсию целевой переменной).

In [13]:
runs = mlflow.search_runs(order_by=["metrics.r2 DESC"])
cols = ['run_id', 'tags.mlflow.runName', 'params.model', 'params.alpha', 'params.l1_ratio',
        'params.max_depth', 'params.min_samples_leaf', 'params.n_estimators',
        'metrics.r2', 'metrics.rmse', 'metrics.mae', 'metrics.train_time_sec']
runs_view = runs[[c for c in cols if c in runs.columns]]
runs_view

,run_id,tags.mlflow.runName,params.model,params.alpha,params.l1_ratio,params.max_depth,params.min_samples_leaf,params.n_estimators,metrics.r2,metrics.rmse,metrics.mae,metrics.train_time_sec
0,71d05e9695704f4db73485e03111b480,forest_200_None,forest,None,None,None,None,200,0.792080,45211.336103,29593.421947,4.875369
1,512c65ea5e144c56b5e12d44fce782b1,forest_200_20,forest,None,None,20,None,200,0.791760,45246.055259,29633.875036,4.543998
2,fb0381ea76964e1faa4ffb4659e8e47a,forest_100_12,forest,None,None,12,None,100,0.777892,46728.470023,31086.913940,1.909736
3,38733e02defb45abacf16e135c6e03f4,tree_12_5,tree,None,None,12,5,None,0.708638,53519.925237,35990.168544,0.155784
4,04133a070e824364970f8ca5f5fda720,tree_8_5,tree,None,None,8,5,None,0.697865,54500.375305,37546.528356,0.059772
5,0c64d78d7cf04ff08a22092f8be5c5ea,tree_20_2,tree,None,None,20,2,None,0.642239,59305.560772,39597.098201,0.103010
6,f27d7d90474e4d4c8c3c48a61abf7232,linear_0.0001_0.15,linear,0.0001,0.15,None,None,None,0.585169,63860.814093,44367.287824,0.016547
7,a6e36df15f7d4a8d8b000b0d953cced2,linear_0.0001_0.001,linear,0.0001,0.001,None,None,None,0.585169,63860.814093,44367.287824,0.009127
8,103b9241ee5f417f98eb1232cdfa20e2,linear_0.001_0.001,linear,0.001,0.001,None,None,None,0.584815,63888.075551,44393.394966,0.013179
9,dbfd470eba3b4e59a3acd14ce12b423b,linear_0.01_0.15,linear,0.01,0.15,None,None,None,0.580420,64225.354587,44719.720037,0.013091


## 6. Обоснование выбора модели

По таблице выше видно чёткую иерархию качества по семействам моделей:

- **линейная модель** даёт самое слабое качество (`r2 ≈ 0.58`) и почти не реагирует на перебор
  гиперпараметров — это означает, что зависимость цены от признаков существенно нелинейна
  (особенно из-за географических координат и их взаимодействия с `median_income`), и линейная модель
  такие закономерности просто не может выучить;
- **дерево решений** заметно лучше (`r2` до `~0.71` при `max_depth=12`), способно учитывать нелинейные
  взаимодействия признаков, но при увеличении глубины (`max_depth=20`) начинает переобучаться — качество
  на валидации падает;
- **случайный лес** — лучший результат (`r2 ≈ 0.79`, `rmse ≈ 45 200$`), так как усреднение по множеству
  деревьев снижает дисперсию (переобучение) одного дерева, сохраняя способность улавливать нелинейные
  зависимости. Увеличение `n_estimators` с 100 до 200 даёт прирост качества, а ограничение `max_depth=20`
  практически не отличается от `max_depth=None` — лес сам неплохо регуляризуется усреднением.

**Вывод:** для разворачивания выбираем лучший запуск случайного леса
(`n_estimators=200`, `max_depth=None`) — он даёт наилучшее соотношение точности предсказания и
устойчивости к переобучению. Платой за это качество является более долгое обучение (секунды против
миллисекунд у линейной модели) и больший размер артефакта модели — для данной задачи (offline-обучение,
сравнительно небольшой датасет) это приемлемый компромисс.

In [14]:
best_run = runs.sort_values("metrics.r2", ascending=False).iloc[0]
best_run_id = best_run['run_id']
model_uri = f"runs:/{best_run_id}/model"

print("Лучший запуск:", best_run['tags.mlflow.runName'])
print("run_id:", best_run_id)
print("model_uri:", model_uri)
print(f"r2={best_run['metrics.r2']:.4f}  rmse={best_run['metrics.rmse']:.1f}  mae={best_run['metrics.mae']:.1f}")

Лучший запуск: forest_200_None
run_id: 71d05e9695704f4db73485e03111b480
model_uri: runs:/71d05e9695704f4db73485e03111b480/model
r2=0.7921  rmse=45211.3  mae=29593.4


## 7. Загрузка лучшей модели и проверка предсказания

Загружаем модель напрямую из MLflow по `model_uri` (`runs:/<run_id>/model`) 

In [15]:
loaded_model = mlflow.sklearn.load_model(model_uri)

sample = X_val[0]
pred_scaled = loaded_model.predict([sample])
pred_price = power_trans.inverse_transform(pred_scaled.reshape(-1, 1))
true_price = power_trans.inverse_transform(y_val[0].reshape(1, -1))

print("Вход (масштабированные признаки):", np.round(sample, 4).tolist())
print(f"Предсказанная цена: {pred_price[0,0]:,.0f} $")
print(f"Истинная цена:      {true_price[0,0]:,.0f} $")

Вход (масштабированные признаки): [1.2977, -1.3266, -0.3507, -0.0581, 0.3673, -0.0182, 0.37, -1.1724, -0.8206]
Предсказанная цена: 127,799 $
Истинная цена:      104,200 $
